# 02 - Preprocessing (KDD stage 2)
Data-gap strategy (proposal risk mitigation):
1. add an explicit **missingness indicator** per gappy sensor column,
2. **interpolate** short gaps (<= 3 days) within each region,
3. fill longer gaps with the region median (the indicator tells the model it is a fill),
4. never interpolate rainfall (an unknown rainfall value cannot be inferred from neighbours).

In [ ]:
import sys
from pathlib import Path

try:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "notebooks" / "common.py").exists())
except StopIteration:
    raise SystemExit("notebooks/common.py not found. Run inside the repo.")
sys.path.insert(0, str(ROOT / "notebooks"))

import pandas as pd
from common import inject_gaps, set_seed, sha256_file, write_meta

set_seed()
df = pd.read_csv(ROOT / "data" / "ewsa_selected.csv", parse_dates=["date"])
df = inject_gaps(df)  # SYNTHETIC ONLY: simulate sensor outages
print(df.isna().mean().round(4)[lambda s: s > 0])

In [ ]:
GAP_COLS = ["river_level_m", "soil_moisture", "humidity_pct"]
df = df.sort_values(["region_id", "date"]).reset_index(drop=True)
for c in GAP_COLS:
    df[f"{c}_missing"] = df[c].isna().astype(int)
    df[c] = df.groupby("region_id")[c].transform(lambda s: s.interpolate(limit=3, limit_direction="both"))
    df[c] = df[c].fillna(df.groupby("region_id")[c].transform("median"))
assert df[GAP_COLS].isna().sum().sum() == 0
print(df[[f"{c}_missing" for c in GAP_COLS]].mean().round(4))

In [ ]:
RANGES = {"rainfall_mm": (0, 1000), "river_level_m": (0, 50), "soil_moisture": (0, 1),
          "temperature_c": (-20, 55), "humidity_pct": (0, 100), "pressure_hpa": (850, 1100),
          "wind_ms": (0, 80)}
for col, (lo, hi) in RANGES.items():
    bad = int(((df[col] < lo) | (df[col] > hi)).sum())
    print(f"{col:15s} out-of-range rows: {bad}")
    df[col] = df[col].clip(lo, hi)

out = ROOT / "data" / "ewsa_clean.csv"
df.to_csv(out, index=False)
write_meta(ROOT / "data" / "ewsa_clean.meta.json", data_kind="synthetic",
           rows=len(df), sha256={out.name: sha256_file(out)})
print("saved", out.name, df.shape)

In [ ]:
{
 "cells": [
  {"cell_type": "markdown", "metadata": {}, "source": "# KDD Stage 2 — Preprocessing\n\nHandles missing data per the proposal's mitigation plan: interpolation for short gaps,\nmulti-source cross-referencing, and explicit missingness indicators as model features\n(never silently imputing without telling the model something was missing)."},
  {"cell_type": "code", "execution_count": null, "metadata": {}, "outputs": [], "source": "import pandas as pd\nimport numpy as np\nimport hashlib, json\nfrom pathlib import Path\n\nRAW = Path(\"../data/raw\")\nPROC = Path(\"../data/processed\")\nPROC.mkdir(parents=True, exist_ok=True)\n\nstations = pd.read_csv(RAW / \"station_readings.csv\", parse_dates=[\"date\"])\nland_cover = pd.read_csv(RAW / \"sanlc_land_cover.csv\")\nprint(stations.isna().sum())"},
  {"cell_type": "code", "execution_count": null, "metadata": {}, "outputs": [], "source": "NUMERIC = [\"rainfall_mm_24h\", \"river_level_m\", \"soil_moisture\", \"temperature_c\", \"humidity_pct\"]\n\ndef clean_region(df: pd.DataFrame) -> pd.DataFrame:\n    df = df.sort_values(\"date\").reset_index(drop=True)\n    # explicit missingness indicators BEFORE imputing (multi-source cross-reference in a real\n    # deployment would fill these from a neighbouring station first; here we interpolate directly)\n    for col in NUMERIC:\n        df[f\"{col}_missing\"] = df[col].isna().astype(int)\n    # short gaps (<=3 days): linear interpolation. Longer gaps: leave NaN -> dropped downstream,\n    # never silently filled with a guess for a long span.\n    df[NUMERIC] = df[NUMERIC].interpolate(method=\"linear\", limit=3, limit_direction=\"both\")\n    # physically impossible values clipped rather than dropped (sensor noise, not garbage rows)\n    df[\"rainfall_mm_24h\"] = df[\"rainfall_mm_24h\"].clip(0, 500)\n    df[\"soil_moisture\"] = df[\"soil_moisture\"].clip(0, 1)\n    df[\"humidity_pct\"] = df[\"humidity_pct\"].clip(0, 100)\n    return df\n\ncleaned = stations.groupby(\"region_id\", group_keys=False).apply(clean_region)\nremaining_gaps = cleaned[NUMERIC].isna().sum().sum()\nprint(\"Remaining NaNs after interpolation (long gaps, dropped next):\", remaining_gaps)\ncleaned = cleaned.dropna(subset=NUMERIC).reset_index(drop=True)\ncleaned.to_parquet(PROC / \"station_readings_clean.parquet\", index=False)\ncleaned.head()"},
  {"cell_type": "code", "execution_count": null, "metadata": {}, "outputs": [], "source": "land_cover.to_parquet(PROC / \"land_cover_clean.parquet\", index=False)\n\ndef sha256_of(path: Path) -> str:\n    h = hashlib.sha256()\n    with open(path, \"rb\") as f:\n        for chunk in iter(lambda: f.read(8192), b\"\"):\n            h.update(chunk)\n    return h.hexdigest()\n\nchecksums = {p.name: sha256_of(p) for p in PROC.glob(\"*\") if p.is_file()}\n(PROC / \"CHECKSUMS.sha256.json\").write_text(json.dumps(checksums, indent=2))\nchecksums"}
 ],
 "metadata": {"kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"}, "language_info": {"name": "python", "version": "3.10"}},
 "nbformat": 4,
 "nbformat_minor": 5
}